# Notebook 3 — API REST con FastAPI + MongoDB + ngrok
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Levantar una API REST que **LEE datos pre-calculados de MongoDB** (no recalcula) y los expone vía HTTP mediante ngrok.

**Prerequisitos:**
- `MONGO_URI` y `NGROK_AUTHTOKEN` configurados en los Secrets de Colab (`🔑`).
- Notebook 1 (Ingesta) y Notebook 2 (SVC) ejecutados y con datos en MongoDB.

**Endpoints implementados:**

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| GET | `/api/salud` | Estado del servidor y conexión a MongoDB |
| GET | `/api/mercado/{ticker}` | Datos OHLCV + indicadores técnicos |
| GET | `/api/svc/{ticker}` | Señal BUY/SELL y métricas del clasificador SVC |
| GET | `/api/rnns/{ticker}` | *(Extra)* Señales y métricas de clasificadores RNN |
| GET | `/api/lstm/{ticker}` | *(Extra)* Pronóstico de precio futuro (regresor LSTM) |
| GET | `/api/noticias` | *(Extra)* Feed de noticias con sentimiento NLP (VADER), filtrable por `fuente` y `sentimiento` |

**Pipeline completo:**
```
yfinance → MongoDB (NB1) → SVC (NB2) → MongoDB → FastAPI (NB3) → Frontend HTML
```


## Paso 1 — Instalación de librerías

In [ ]:
# ============================================================
# PASO 1: Instalar todas las dependencias necesarias
# ============================================================
!pip install fastapi uvicorn pyngrok nest-asyncio 'pymongo[srv]' --quiet

print("✓ Librerías instaladas correctamente")
print("  fastapi    — framework web para la API REST")
print("  uvicorn    — servidor ASGI para ejecutar FastAPI")
print("  pyngrok    — túnel ngrok para exponer la API a Internet")
print("  nest-asyncio — permite asyncio dentro de Jupyter/Colab")
print("  pymongo    — cliente oficial de MongoDB para Python")


## Paso 2 — Importaciones y configuración global

In [ ]:
# ============================================================
# PASO 2: Importaciones y configuración global
# ============================================================
import json
import time
import threading
import warnings
from datetime import datetime
from typing import Any, Dict, List, Optional

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, ServerSelectionTimeoutError
from pyngrok import conf, ngrok

# Silenciar advertencias menores
warnings.filterwarnings('ignore')

# Aplicar nest_asyncio para que uvicorn funcione dentro de Colab/Jupyter
nest_asyncio.apply()

print("✓ Importaciones completadas")


## Paso 3 — Configurar credenciales (Secrets de Colab)

In [ ]:
# ============================================================
# PASO 3: Leer MONGO_URI y NGROK_AUTHTOKEN desde los Secrets
# de Colab (ícono de llave 🔑 en el panel izquierdo).
#
# IMPORTANTE: nunca pegues credenciales directamente en el
# código; usa siempre los Secrets para proteger tus datos.
# ============================================================
from google.colab import userdata

# ── ngrok: token de autenticación ────────────────────────────
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
conf.get_default().auth_token = NGROK_TOKEN  # pyngrok 8.x
print("✓ ngrok configurado")

# ── MongoDB: URI de conexión ──────────────────────────────────
MONGO_URI = userdata.get('MONGO_URI')

# ── Nombre de la base de datos ────────────────────────────────
DB_NAME  = 'spbi'

# ── Colecciones que usa la API (solo lectura) ─────────────────
COL_PRECIOS     = 'precios_ohlcv'       # Notebook 1
COL_PREDICCIONES = 'predicciones'       # Notebook 2
COL_METRICAS    = 'metricas_modelos'    # Notebook 2
COL_NOTICIAS    = 'noticias'            # Notebook 8 (NLP / VADER)
# COL_RNN ya no se usa: el LSTM comparte 'predicciones' y 'metricas_modelos' con el SVC (filtrando modelo='LSTM')

print("✓ Variables de configuración definidas")
print(f"  Base de datos : {DB_NAME}")
print(f"  Colecciones   : {COL_PRECIOS}, {COL_PREDICCIONES}, {COL_METRICAS}, {COL_NOTICIAS}")


## Paso 4 — Conexión a MongoDB Atlas

In [ ]:
# ============================================================
# PASO 4: Conectar a MongoDB Atlas y verificar la conexión
# ============================================================

def crear_cliente_mongo(uri: str, timeout_ms: int = 5000) -> MongoClient:
    """
    Crea un cliente MongoDB con timeout configurable.
    Lanza ConnectionError si no se puede conectar.
    """
    cliente = MongoClient(uri, serverSelectionTimeoutMS=timeout_ms)
    # Ping rápido para verificar que la conexión es válida
    cliente.admin.command('ping')
    return cliente

try:
    cliente_mongo = crear_cliente_mongo(MONGO_URI)
    db = cliente_mongo[DB_NAME]

    # Contar documentos en cada colección para verificar que hay datos
    n_precios  = db[COL_PRECIOS].count_documents({})
    n_pred     = db[COL_PREDICCIONES].count_documents({})
    n_met      = db[COL_METRICAS].count_documents({})
    n_rnn      = db[COL_PREDICCIONES].count_documents({'modelo': 'LSTM'})
    n_noticias = db[COL_NOTICIAS].count_documents({})

    print("✓ Conexión a MongoDB Atlas exitosa")
    print(f"  Base de datos       : {DB_NAME}")
    print(f"  {COL_PRECIOS:<22}: {n_precios:,} documentos")
    print(f"  {COL_PREDICCIONES:<22}: {n_pred:,} documentos")
    print(f"  {COL_METRICAS:<22}: {n_met:,} documentos")
    print(f"  predicciones (LSTM)  : {n_rnn:,} documentos")
    print(f"  {COL_NOTICIAS:<22}: {n_noticias:,} documentos")

    if n_precios == 0:
        print("\n⚠  AVISO: 'precios_ohlcv' vacío — ejecuta el Notebook 1 primero")
    if n_pred == 0:
        print("⚠  AVISO: 'predicciones' vacío — ejecuta el Notebook 2 primero")
    if n_noticias == 0:
        print("⚠  AVISO: 'noticias' vacío — ejecuta el Notebook 8 (NLP) para el módulo M5")

except (ConnectionFailure, ServerSelectionTimeoutError) as e:
    raise RuntimeError(
        f"❌ No se pudo conectar a MongoDB Atlas.\n"
        f"   Verifica que MONGO_URI sea correcto y que el IP 0.0.0.0/0 "
        f"esté en la lista de acceso de red en Atlas.\n"
        f"   Error: {e}"
    )


## Paso 5 — Crear la aplicación FastAPI

In [ ]:
# ============================================================
# PASO 5: Crear la aplicación FastAPI con CORS habilitado
#
# CORS (Cross-Origin Resource Sharing) es necesario para que
# el frontend HTML (desplegado en GitHub Pages u otro dominio)
# pueda hacer fetch() a esta API sin ser bloqueado por el
# navegador.
# ============================================================

app = FastAPI(
    title="Ernesto Investing AI — API REST",
    description=(
        "API que expone datos de mercado y predicciones del "
        "clasificador SVC para los 5 activos mineros del proyecto. "
        "Lee datos pre-calculados de MongoDB (no recalcula en tiempo real)."
    ),
    version="1.0.0",
    docs_url="/docs",      # Swagger UI
    redoc_url="/redoc",    # ReDoc (documentación alternativa)
)

# Habilitar CORS para que cualquier origen pueda consumir la API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # Cualquier origen (frontend en GitHub Pages, etc.)
    allow_credentials=True,
    allow_methods=["*"],          # GET, POST, OPTIONS, etc.
    allow_headers=["*"],          # Incluyendo ngrok-skip-browser-warning
)

# ── Tickers soportados por el sistema ────────────────────────
TICKERS_VALIDOS = {'FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP'}

def validar_ticker(ticker: str) -> str:
    """
    Normaliza el ticker a mayúsculas y verifica que esté
    en la lista de activos soportados.
    """
    t = ticker.upper()
    if t not in TICKERS_VALIDOS:
        raise HTTPException(
            status_code=404,
            detail={
                "error": f"Ticker '{t}' no está en el sistema.",
                "tickers_disponibles": sorted(TICKERS_VALIDOS),
                "sugerencia": "Verifica el símbolo del ticker (distingue mayúsculas/minúsculas en Yahoo Finance)."
            }
        )
    return t

print("✓ Aplicación FastAPI creada")
print(f"  Tickers soportados: {sorted(TICKERS_VALIDOS)}")


## Paso 6 — Endpoint `/api/salud`

In [ ]:
# ============================================================
# PASO 6: Endpoint de salud del sistema
#
# Verifica que:
# 1. El servidor FastAPI está levantado y respondiendo.
# 2. La conexión a MongoDB sigue activa.
# 3. Las colecciones tienen datos (Notebooks 1 y 2 ejecutados).
# ============================================================

@app.get(
    "/api/salud",
    summary="Estado del servidor",
    tags=["Sistema"],
    response_description="Estado operativo del servidor y la base de datos"
)
def salud():
    """
    Verifica el estado del servidor y la conexión a MongoDB.

    Retorna:
    - **estado**: 'operativo' o 'degradado'
    - **base_datos**: 'conectada' o 'error'
    - Conteo de documentos en cada colección
    """
    try:
        # Ping a MongoDB para verificar que la conexión sigue viva
        cliente_mongo.admin.command("ping")

        n_precios = db[COL_PRECIOS].count_documents({})
        n_pred    = db[COL_PREDICCIONES].count_documents({})
        n_met     = db[COL_METRICAS].count_documents({})

        # Si faltan datos, el estado es 'degradado' pero sigue funcionando
        datos_completos = (n_precios > 0 and n_pred > 0 and n_met > 0)

        return {
            "estado"               : "operativo" if datos_completos else "degradado",
            "base_datos"           : "conectada",
            "colecciones": {
                COL_PRECIOS      : n_precios,
                COL_PREDICCIONES : n_pred,
                COL_METRICAS     : n_met,
            },
            "tickers_disponibles"  : sorted(TICKERS_VALIDOS),
            "timestamp"            : datetime.now().isoformat(),
            "version_api"          : "1.0.0",
            "advertencia"          : None if datos_completos else
                                     "Una o más colecciones están vacías. Ejecuta los Notebooks 1 y 2."
        }

    except Exception as e:
        # Si MongoDB no responde, retorna 503 (servicio no disponible)
        raise HTTPException(
            status_code=503,
            detail={
                "estado"     : "error",
                "base_datos" : "desconectada",
                "error"      : str(e),
                "timestamp"  : datetime.now().isoformat()
            }
        )

print("✓ Endpoint /api/salud registrado")


## Paso 7 — Endpoint `/api/mercado/{ticker}`

In [ ]:
# ============================================================
# PASO 7: Endpoint de datos de mercado
#
# Lee de la colección 'precios_ohlcv' los datos OHLCV e
# indicadores técnicos calculados por el Notebook 1.
#
# El parámetro 'dias' permite solicitar los últimos N días
# (por defecto: todos los disponibles).
# ============================================================

@app.get(
    "/api/mercado/{ticker}",
    summary="Datos OHLCV e indicadores técnicos",
    tags=["Mercado"],
    response_description="Serie histórica de precios con SMA, EMA y RSI"
)
def mercado(
    ticker: str,
    dias: Optional[int] = Query(
        default=None,
        ge=1,
        le=1500,
        description="Últimos N días de datos. Sin valor → devuelve todo el histórico disponible."
    )
):
    """
    Retorna datos OHLCV + indicadores técnicos de un ticker.

    **Indicadores disponibles:** SMA-20, SMA-50, EMA-12, EMA-26, RSI-14, MACD, Bandas de Bollinger.

    **Parámetro opcional:** `?dias=90` para los últimos 90 días.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)

    # Campos a excluir de la respuesta de MongoDB
    proyeccion = {"_id": 0, "created_at": 0}

    # Recuperar documentos ordenados cronológicamente
    cursor = db[COL_PRECIOS].find({"ticker": t}, proyeccion).sort("fecha", 1)
    datos  = list(cursor)

    if not datos:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay datos de mercado para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook 1 (Ingesta) para poblar la colección precios_ohlcv.",
                "ticker"     : t
            }
        )

    # Aplicar filtro de días si se especificó
    if dias is not None:
        datos = datos[-dias:]

    # Calcular rango de fechas disponibles
    fecha_inicio = datos[0].get("fecha",  "—")
    fecha_fin    = datos[-1].get("fecha", "—")
    ultimo_precio = datos[-1].get("close", None)

    return {
        "ticker"           : t,
        "empresa"          : datos[-1].get("company_name", t),
        "registros"        : len(datos),
        "fecha_inicio"     : fecha_inicio,
        "fecha_fin"        : fecha_fin,
        "ultimo_precio"    : ultimo_precio,
        "indicadores"      : ["sma_20", "sma_50", "ema_12", "ema_26", "rsi_14",
                               "macd_line", "macd_signal", "macd_histogram",
                               "bb_upper", "bb_middle", "bb_lower"],
        "datos"            : datos
    }

print("✓ Endpoint /api/mercado/{ticker} registrado")


## Paso 8 — Endpoint `/api/svc/{ticker}`

In [ ]:
# ============================================================
# PASO 8: Endpoint del clasificador SVC
#
# Lee de las colecciones 'predicciones' y 'metricas_modelos'
# los resultados generados por el Notebook 2 (SVC).
#
# La API NO recalcula: solo sirve lo que ya está en MongoDB.
# Esto garantiza respuestas en milisegundos.
# ============================================================

@app.get(
    "/api/svc/{ticker}",
    summary="Señal BUY/SELL y métricas del clasificador SVC",
    tags=["Clasificador SVC"],
    response_description="Predicción actual, métricas del modelo y matriz de confusión"
)
def svc(ticker: str):
    """
    Retorna la señal de trading actual (BUY/SELL) y las métricas
    de evaluación del clasificador SVC para un ticker.

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por el Notebook 2.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción actual ──────────────────────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": "SVC"},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay predicción SVC para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook 2 (SVC) para generar predicciones.",
                "ticker"     : t
            }
        )

    # ── 2. Métricas del modelo ────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": "SVC"},
        {**proyeccion, "fecha_entrenamiento": 0}
    )

    return {
        "ticker"            : t,
        "modelo"            : "SVC",
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "tipo_features"     : prediccion.get("tipo_features"),
        "prediccion": {
            "senal"      : prediccion.get("senal"),
            "confianza"  : prediccion.get("confianza"),
        },
        "metricas": {
            "accuracy"          : metricas.get("accuracy")   if metricas else None,
            "precision"         : metricas.get("precision")  if metricas else None,
            "recall"            : metricas.get("recall")     if metricas else None,
            "f1"                : metricas.get("f1")         if metricas else None,
            "matriz_confusion"  : metricas.get("matriz_confusion") if metricas else None,
            "mejor_kernel"      : metricas.get("mejor_kernel")     if metricas else None,
            "mejor_C"           : metricas.get("mejor_C")          if metricas else None,
            "mejor_gamma"       : metricas.get("mejor_gamma")      if metricas else None,
        },
        "historico_senales" : prediccion.get("historico_senales", [])
    }

print("✓ Endpoint /api/svc/{ticker} registrado (campos alineados con el Notebook 2 real)")


## Paso 9 — Endpoint `/api/rnns/{ticker}` *(Extra — clasificador LSTM)*

In [ ]:
# ============================================================
# PASO 9 (EXTRA): Endpoint para los clasificadores RNN
# (LSTM / BiLSTM / GRU / SimpleRNN)
#
# Lee de las MISMAS colecciones que el SVC ('predicciones' y
# 'metricas_modelos'), filtrando por el campo 'modelo' que el
# usuario elige con el parámetro de query ?modelo=. Se mantiene
# un solo patrón de colecciones para todas las arquitecturas,
# en vez de una colección aparte por modelo.
#
# Si no hay documentos con el modelo pedido para el ticker,
# retorna 404 con un mensaje claro en vez de crashear el servidor.
# ============================================================

MODELOS_RNN_VALIDOS = {'LSTM', 'BiLSTM', 'GRU', 'SimpleRNN'}

@app.get(
    "/api/rnns/{ticker}",
    summary="Señal BUY/SELL y métricas de un clasificador RNN (LSTM/BiLSTM/GRU/SimpleRNN)",
    tags=["Clasificadores RNN"],
    response_description="Predicción actual, métricas del modelo, matriz de confusión e historial de épocas"
)
def rnns(ticker: str, modelo: str = "LSTM"):
    """
    Retorna la señal de trading actual (BUY/SELL) y las métricas
    de evaluación de la arquitectura RNN elegida para un ticker.

    **Parámetro `modelo`:** uno de `LSTM`, `BiLSTM`, `GRU`, `SimpleRNN`
    (por defecto `LSTM`). Corresponde al selector de arquitectura del
    frontend (Módulo M3).

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por los Notebooks 3 (LSTM), 4 (BiLSTM), 5 (GRU) y 6 (SimpleRNN),
    filtrando por el campo `modelo`.

    Este endpoint es opcional: si el notebook correspondiente no ha sido
    ejecutado para este ticker/modelo, retorna 404 con un mensaje descriptivo.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    m = modelo.strip()

    if m not in MODELOS_RNN_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error"             : f"Modelo '{m}' no reconocido.",
                "modelos_validos"   : sorted(MODELOS_RNN_VALIDOS),
                "sugerencia"        : "Usa ?modelo=LSTM, ?modelo=BiLSTM, ?modelo=GRU o ?modelo=SimpleRNN."
            }
        )

    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción actual ──────────────────────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": m},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay predicción {m} para '{t}'.",
                "sugerencia" : f"Ejecuta el notebook del clasificador {m} para este ticker.",
                "ticker"     : t,
                "modelo"     : m
            }
        )

    # ── 2. Métricas del modelo ────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": m},
        {**proyeccion, "fecha_entrenamiento": 0}
    )

    return {
        "ticker"            : t,
        "modelo"            : m,
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "n_steps"           : prediccion.get("n_steps"),
        "prediccion": {
            "senal"      : prediccion.get("senal"),
            "confianza"  : prediccion.get("confianza"),
        },
        "metricas": {
            "accuracy"          : metricas.get("accuracy")   if metricas else None,
            "precision"         : metricas.get("precision")  if metricas else None,
            "recall"            : metricas.get("recall")     if metricas else None,
            "f1"                : metricas.get("f1")         if metricas else None,
            "epocas_entrenadas" : metricas.get("epocas_entrenadas") if metricas else None,
            "matriz_confusion"  : metricas.get("matriz_confusion")  if metricas else None,
            "historial_epocas"  : metricas.get("historial_epocas")  if metricas else None,
        },
        "historico_senales" : prediccion.get("historico_senales", [])
    }

print("✓ Endpoint /api/rnns/{ticker}?modelo=LSTM|BiLSTM|GRU|SimpleRNN registrado")


## Paso 10 — Endpoint `/api/lstm/{ticker}` *(Extra — regresor de precios)*

In [ ]:
# ============================================================
# PASO 10 (EXTRA): Endpoint del regresor LSTM (precio futuro)
#
# Lee de las MISMAS colecciones que los clasificadores
# ('predicciones' y 'metricas_modelos'), filtrando por
# modelo='LSTM_REG'. Este endpoint es de REGRESIÓN, no de
# clasificación: devuelve un precio continuo proyectado a
# 7/14/30/60 días, con banda de confianza, en vez de una
# señal BUY/SELL.
#
# Traduce el esquema interno de Mongo (proyecciones_horizonte
# con claves '7d'/'14d'/'30d'/'60d') al esquema que consume
# el frontend (prediccion_futura con claves '7_dias'/etc.).
# ============================================================

HORIZONTES_DISPONIBLES = [7, 14, 30, 60]

@app.get(
    "/api/lstm/{ticker}",
    summary="Pronóstico de precio futuro (regresor LSTM)",
    tags=["Regresor LSTM"],
    response_description="Precio actual, histórico real vs. predicho, y proyección a futuro con bandas de confianza"
)
def lstm_regresor(ticker: str, horizonte: int = 60):
    """
    Retorna el pronóstico de precio futuro del regresor LSTM para un ticker.

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por el Notebook del Regresor LSTM (`modelo='LSTM_REG'`).

    El parámetro `horizonte` se acepta por compatibilidad con el frontend
    (slider de 7/14/30/60 días), pero la API siempre devuelve las 4
    proyecciones disponibles en `prediccion_futura`; el frontend elige
    cuál mostrar según el horizonte seleccionado.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción + proyecciones a futuro ─────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": "LSTM_REG"},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay pronóstico LSTM_REG para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook del Regresor LSTM para este ticker.",
                "ticker"     : t
            }
        )

    # ── 2. Métricas del modelo ─────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": "LSTM_REG"},
        {**proyeccion, "fecha_entrenamiento": 0}
    ) or {}

    # ── 3. Traducir 'proyecciones_horizonte' (esquema interno) a
    #        'prediccion_futura' (esquema esperado por el frontend) ──
    horizonte_interno = prediccion.get("proyecciones_horizonte", {})
    prediccion_futura = {}
    for h in HORIZONTES_DISPONIBLES:
        datos_h = horizonte_interno.get(f"{h}d")
        if datos_h:
            prediccion_futura[f"{h}_dias"] = {
                "precio"    : datos_h.get("precio_estimado"),
                "banda_sup" : datos_h.get("banda_superior"),
                "banda_inf" : datos_h.get("banda_inferior"),
            }

    return {
        "ticker"            : t,
        "modelo"            : "LSTM_REG",
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "n_steps"           : prediccion.get("n_steps"),
        "precio_actual_usd" : prediccion.get("ultimo_precio"),
        "metricas": {
            "rmse_usd"           : metricas.get("rmse_usd"),
            "rmse_pct"           : metricas.get("rmse_pct"),
            "mae_usd"            : metricas.get("mae_usd"),
            "r2"                 : metricas.get("r2"),
            "rmse_arima_baseline": metricas.get("rmse_arima_baseline"),
            "epocas_entrenadas"  : metricas.get("epocas_entrenadas"),
            "historial_epocas"   : metricas.get("historial_epocas"),
        },
        "historico_precios" : prediccion.get("historico_predicciones", []),
        "serie_diaria_60d"  : prediccion.get("serie_diaria_60d", []),
        "prediccion_futura" : prediccion_futura
    }

print("✓ Endpoint /api/lstm/{ticker} registrado (regresor de precios, modelo='LSTM_REG')")


## Paso 11 — Endpoint `/api/noticias` *(Extra — sentimiento NLP)*


In [ ]:
# ============================================================
# PASO 11 (EXTRA): Endpoint del feed de noticias con sentimiento NLP
#
# Lee la colección 'noticias' poblada por el Notebook 8 (VADER).
# Replica EXACTAMENTE los dos filtros que expone el selector del
# Módulo M5 del frontend:
#   - fuente:      'all' | 'bloomberg' | 'cnbc' | 'reuters'
#   - sentimiento: 'all' | 'bullish' | 'bearish' | 'neutral'
#
# Los <select> del HTML mandan los valores en minúsculas, pero
# MongoDB guarda 'fuente' con mayúscula inicial (Bloomberg, CNBC,
# Reuters, MarketWatch) y 'sentimiento' siempre en mayúsculas
# (BULLISH/BEARISH/NEUTRAL). Por eso la comparación de 'fuente' es
# case-insensitive (regex) y 'sentimiento' se normaliza a upper().
# ============================================================

COL_NOTICIAS_TAG = "NLP"

FUENTES_VALIDAS = {'bloomberg', 'cnbc', 'reuters', 'marketwatch'}
SENTIMIENTOS_VALIDOS = {'bullish', 'bearish', 'neutral'}

@app.get(
    "/api/noticias",
    summary="Feed de noticias con sentimiento NLP (VADER)",
    tags=["NLP"],
    response_description="Noticias filtradas por fuente y sentimiento, con sentimiento consolidado del conjunto"
)
def noticias(
    fuente: str = Query(
        default="all",
        description="'all', 'bloomberg', 'cnbc' o 'reuters' (case-insensitive, igual que el <select> del frontend)"
    ),
    sentimiento: str = Query(
        default="all",
        description="'all', 'bullish', 'bearish' o 'neutral'"
    ),
    ticker: Optional[str] = Query(
        default=None,
        description="Opcional: filtra por un ticker específico (el Módulo M5 no lo usa hoy, pero queda disponible)"
    ),
    limite: int = Query(
        default=50, ge=1, le=200,
        description="Máximo de noticias a devolver, ordenadas por fecha de publicación descendente"
    )
):
    """
    Retorna el feed de noticias procesado por el Notebook 8 (VADER),
    con los mismos filtros que usan los selectores del Módulo M5:

    - **fuente**: `all` | `bloomberg` | `cnbc` | `reuters`
    - **sentimiento**: `all` | `bullish` | `bearish` | `neutral`

    **Fuente de datos:** colección `noticias` de MongoDB, poblada por el
    Notebook 8 (VADER sobre titulares de yfinance + respaldo simulado).

    **Importante:** si el filtro no tiene coincidencias (ej. no hay noticias
    'Reuters' + 'Bearish'), la respuesta es `200 OK` con `"noticias": []` y
    `"mensaje"` explicando que no hay resultados — NO se lanza un 404. Un
    404 solo ocurre si la colección `noticias` está completamente vacía
    (el Notebook 8 nunca se ejecutó).

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    query: Dict[str, Any] = {}

    # ── Filtro de ticker (opcional) ─────────────────────────────
    if ticker:
        query["ticker"] = validar_ticker(ticker)

    # ── Filtro de fuente (case-insensitive) ─────────────────────
    f = fuente.strip().lower()
    if f != "all":
        if f not in FUENTES_VALIDAS:
            raise HTTPException(
                status_code=400,
                detail={
                    "error": f"Fuente '{fuente}' no reconocida.",
                    "fuentes_validas": sorted(FUENTES_VALIDAS) + ["all"]
                }
            )
        query["fuente"] = {"$regex": f"^{f}$", "$options": "i"}

    # ── Filtro de sentimiento ────────────────────────────────────
    s = sentimiento.strip().lower()
    if s != "all":
        if s not in SENTIMIENTOS_VALIDOS:
            raise HTTPException(
                status_code=400,
                detail={
                    "error": f"Sentimiento '{sentimiento}' no reconocido.",
                    "sentimientos_validos": sorted(SENTIMIENTOS_VALIDOS) + ["all"]
                }
            )
        query["sentimiento"] = s.upper()  # BULLISH / BEARISH / NEUTRAL

    # ── ¿La colección está vacía por completo? (Notebook 8 nunca ejecutado) ──
    # Esto SÍ es un error real de configuración, y se distingue de "el filtro
    # elegido no tiene coincidencias", que es un resultado válido, no un error.
    total_coleccion = db[COL_NOTICIAS].estimated_document_count()
    if total_coleccion == 0:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : "La colección 'noticias' está vacía (no hay ningún documento).",
                "sugerencia" : "Ejecuta el Notebook 8 (NLP) para poblar la colección 'noticias'."
            }
        )

    proyeccion = {"_id": 0, "created_at": 0}
    cursor = db[COL_NOTICIAS].find(query, proyeccion).sort("fecha_publicacion", -1).limit(limite)
    docs = list(cursor)

    # ── Sentimiento consolidado del conjunto filtrado ────────────
    # Si el filtro no tiene coincidencias, docs=[] y esto se queda en None;
    # NO se lanza 404: un filtro sin resultados es una respuesta 200 válida,
    # y así debe interpretarlo el frontend (mostrar "sin noticias", no
    # "sin conexión").
    compounds = [d.get("compound", 0.0) for d in docs]
    compound_promedio = round(sum(compounds) / len(compounds), 4) if compounds else None
    etiqueta_consolidada = None
    if compound_promedio is not None:
        if compound_promedio > 0.05:
            etiqueta_consolidada = "BULLISH"
        elif compound_promedio < -0.05:
            etiqueta_consolidada = "BEARISH"
        else:
            etiqueta_consolidada = "NEUTRAL"

    return {
        "total"                    : len(docs),
        "filtro_aplicado"          : {"fuente": fuente, "sentimiento": sentimiento, "ticker": ticker},
        "compound_promedio"        : compound_promedio,
        "sentimiento_consolidado"  : etiqueta_consolidada,
        "mensaje"                  : None if docs else "No hay noticias que coincidan con el filtro seleccionado.",
        "noticias"                 : docs
    }

print("✓ Endpoint /api/noticias?fuente=&sentimiento=&ticker=&limite= registrado")


## Paso 12 — Levantar servidor y exponer con ngrok

In [ ]:
# ============================================================
# PASO 12: Levantar el servidor FastAPI y el túnel ngrok
#
# Secuencia correcta de arranque:
#   1. pkill -9 ngrok  → matar proceso a nivel OS
#   2. time.sleep(3)   → esperar que el proceso muera
#   3. ngrok.kill()    → limpiar estado interno de pyngrok
#   4. time.sleep(2)   → esperar que ngrok.io libere el túnel
#   5. ngrok.connect() → abrir túnel nuevo y limpio
#
# ⚠ No mover estas líneas ni cambiar el orden.
# ⚠ NO abrir un segundo túnel en ninguna otra celda.
# ============================================================

# ── Paso 10.1: limpiar procesos ngrok anteriores ─────────────
!pkill -9 ngrok 2>/dev/null || true
time.sleep(3)

try:
    ngrok.kill()
except Exception:
    pass
time.sleep(2)

# ── Paso 10.2: abrir el túnel ngrok ──────────────────────────
tunel = ngrok.connect(8000)
URL_PUBLICA = tunel.public_url

print("=" * 68)
print(f"  URL PÚBLICA DE LA API : {URL_PUBLICA}")
print("=" * 68)
print()
print(f"  ✓  Salud   :  {URL_PUBLICA}/api/salud")
print(f"  ✓  Mercado :  {URL_PUBLICA}/api/mercado/BVN")
print(f"  ✓  SVC     :  {URL_PUBLICA}/api/svc/BVN")
print(f"  ✓  RNNs    :  {URL_PUBLICA}/api/rnns/BVN")
print(f"  ✓  LSTM    :  {URL_PUBLICA}/api/lstm/BVN")
print(f"  ✓  Noticias:  {URL_PUBLICA}/api/noticias?fuente=all&sentimiento=all")
print(f"  ✓  Swagger :  {URL_PUBLICA}/docs")
print()
print("  >>> COPIA LA URL PÚBLICA DE ARRIBA <<<")
print("  >>> Pégala en index.html del frontend (campo API_URL) <<<")
print()
print("  ⚠  Mantén este notebook corriendo mientras uses el frontend.")

# ── Paso 10.3: arrancar uvicorn en un hilo aparte ────────────
def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo al servidor para iniciar

print()
print("  ✓ Servidor FastAPI activo en el puerto 8000")
print("  ✓ API lista para recibir solicitudes del frontend")


## Paso 13 — Verificación final del sistema

In [ ]:
# ============================================================
# PASO 13: Verificar que los endpoints principales responden
# correctamente antes de conectar el frontend.
#
# Ejecuta esta celda DESPUÉS del Paso 10.
# ============================================================

import requests

TICKERS_PRUEBA = ['FSM', 'BVN']
CABECERAS = {'ngrok-skip-browser-warning': '1'}  # Evita la pantalla de advertencia de ngrok

print("=" * 68)
print("VERIFICACIÓN FINAL — ENDPOINTS DE LA API")
print("=" * 68)

# ── 1. /api/salud ─────────────────────────────────────────────
print("\n1️⃣  /api/salud")
try:
    r = requests.get(f"{URL_PUBLICA}/api/salud", headers=CABECERAS, timeout=10)
    d = r.json()
    estado_bd = d.get('base_datos', '—')
    estado    = d.get('estado', '—')
    col_str   = ", ".join(f"{k}: {v}" for k,v in d.get('colecciones', {}).items())
    print(f"   Estado: {estado} | BD: {estado_bd}")
    print(f"   Colecciones — {col_str}")
    if d.get('advertencia'):
        print(f"   ⚠  {d['advertencia']}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 2. /api/mercado/{ticker} ──────────────────────────────────
print(f"\n2️⃣  /api/mercado/{{ticker}}")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/mercado/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            print(f"   ✓ {ticker:<12} — {d['registros']} registros | "
                  f"último precio: ${d.get('ultimo_precio', '—')} | "
                  f"rango: {d['fecha_inicio']} → {d['fecha_fin']}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 3. /api/svc/{ticker} ─────────────────────────────────────
print(f"\n3️⃣  /api/svc/{{ticker}}")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/svc/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            senal = d.get('prediccion', {}).get('senal_actual', '—')
            acc   = d.get('metricas',   {}).get('accuracy', None)
            acc_s = f"{acc*100:.1f}%" if acc is not None else "—"
            print(f"   ✓ {ticker:<12} — Señal: {senal:<5} | Accuracy: {acc_s}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 4. /api/noticias ──────────────────────────────────────────
print(f"\n4️⃣  /api/noticias")
try:
    r = requests.get(f"{URL_PUBLICA}/api/noticias", params={"fuente": "all", "sentimiento": "all"},
                      headers=CABECERAS, timeout=10)
    if r.status_code == 200:
        d = r.json()
        print(f"   ✓ total={d.get('total','—')} | sentimiento consolidado: {d.get('sentimiento_consolidado','—')} "
              f"(compound={d.get('compound_promedio','—')})")
    else:
        print(f"   ⚠ HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 5. Swagger UI ────────────────────────────────────────────
print(f"\n5️⃣  Documentación interactiva (Swagger UI)")
print(f"   🔗 {URL_PUBLICA}/docs")

print("\n" + "=" * 68)
print("  ✅ Verificación completa")
print(f"  🔗 Pega esta URL en index.html del frontend:")
print(f"     const API_URL = \"{URL_PUBLICA}\";")
print("=" * 68)


## ⚠ Importante — Mientras el frontend esté en uso

- **Mantén este notebook corriendo** en Google Colab. Si se cierra o el runtime
  se desconecta, el túnel ngrok muere y el frontend mostrará "Error API".
- **La URL de ngrok cambia cada vez** que reinicias el kernel o el túnel.
  Cuando eso pase, copia la nueva URL del Paso 10 y actualiza la constante
  `API_URL` en `index.html`.
- Para la **exposición final**, considera el bono Streamlit: ofrece una URL
  permanente que no depende de que Colab esté corriendo.
